In [ ]:
!pip install -q transformers datasets accelerate

from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Finetuning"
DATA_DIR = os.path.join(BASE_DIR, "data/tokenized")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"--> Directives established.\nData Source: {DATA_DIR}\nCheckpoint target: {CHECKPOINT_DIR}")

Mounted at /content/drive
--> Directives established.
Data Source: /content/drive/MyDrive/Finetuning/data/tokenized
Checkpoint target: /content/drive/MyDrive/Finetuning/output/checkpoints


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_tokenized_json(file_name):
    full_path = os.path.join(DATA_DIR, file_name)
    print(f"Reading {file_name} from Drive...")

    with open(full_path, 'r') as f:
        data = json.load(f)

    if isinstance(data, dict):
        key = "input_ids" if "input_ids" in data else list(data.keys())[0]
        payload = data[key]
    else:
        payload = data

    if isinstance(payload, list) and len(payload) > 0 and isinstance(payload[0], int):
        return Dataset.from_dict({"input_ids": [payload]})
    else:
        return Dataset.from_dict({"input_ids": payload})

raw_datasets = DatasetDict({
    "train": load_tokenized_json("train_tokens.json"),
    "validation": load_tokenized_json("val_tokens.json")
})

print("\nIngestion complete. Current dataset footprint:")
print(raw_datasets)

Reading train_tokens.json from Drive...
Reading val_tokens.json from Drive...

Ingestion complete. Current dataset footprint:
DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 148346
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 18543
    })
})


In [ ]:
block_size = 512

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()
    return result

print("Slicing raw tokens into fixed context windows...")
lm_datasets = raw_datasets.map(
    group_texts,
    batched=True,
    num_proc=2,
)

print("\nChunking complete. Data shapes prepared for model ingestion:")
print(lm_datasets)

Slicing raw tokens into fixed context windows...


Map (num_proc=2):   0%|          | 0/148346 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/18543 [00:00<?, ? examples/s]


Chunking complete. Data shapes prepared for model ingestion:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 148346
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 18543
    })
})


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling
import torch

print("Loading base pre-trained GPT-2 Small weights...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Model and Collator initialization complete.")

Loading base pre-trained GPT-2 Small weights...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and Collator initialization complete.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,                   # Write checkpoint to Google Drive every 200 steps
    save_total_limit=2,               # Keep only the 2 latest checkpoints to avoid clogging Drive storage
    learning_rate=5e-5,               # Safe standard starting LR for full-parameter adjustments
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # Simulates batch size 16 calculation states before weight changes
    fp16=True,                        # Enables mixed-precision mathematical optimization
    logging_steps=50,                 # Print loss readouts frequently to track stability
    num_train_epochs=1,               # Let's run a single epoch baseline check first
    report_to="none"                  # Keep outputs native to Colab for simplicity
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)

In [ ]:
import glob

checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))

if checkpoints:
    latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    print(f"--> Found active save state. Resuming execution from checkpoint: {latest_checkpoint}")
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("--> No prior execution states detected. Commencing fresh fine-tuning deployment...")
    trainer.train()

print("\nTraining run completed. Securing permanent final weights...")
final_model_path = os.path.join(OUTPUT_DIR, "final_screenplay_gpt2")

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"--> Success. Isolated, production-ready weights written cleanly to: {final_model_path}")

--> Found active save state. Resuming execution from checkpoint: /content/drive/MyDrive/Finetuning/output/checkpoints/checkpoint-5600


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
5800,1.393919,1.327636
6000,1.452174,1.327878
6200,1.435192,1.326494
6400,1.424345,1.325840
6600,1.423755,1.324502
6800,1.379429,1.322827
7000,1.449911,1.324011
7200,1.443793,1.322343
7400,1.421113,1.323397
7600,1.394484,1.321109


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training run completed. Securing permanent final weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--> Success. Isolated, production-ready weights written cleanly to: /content/drive/MyDrive/Finetuning/output/final_screenplay_gpt2


In [ ]:
import pandas as pd
history = pd.DataFrame(trainer.state.log_history)
history.to_csv("/content/drive/MyDrive/Finetuning/output/cuda_benchmarks.csv", index=False)

In [ ]:
import os
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

MODEL_PATH = "/content/drive/MyDrive/Finetuning/output/final_screenplay_gpt2"

print(f"--> Loading custom full-parameter model from {MODEL_PATH}...")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"--> Model successfully loaded onto {device.upper()}.")

def generate_scene(prompt_text, max_len=300, temp=0.85):
    print(f"\n[PROMPT]: {prompt_text}\n" + "="*50)

    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    # Generate the sequence
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_len,
        temperature=temp,             # 0.85 gives good creative variance without going crazy
        top_k=50,                     # Limits vocabulary to top 50 likely next words
        top_p=0.92,                   # Nucleus sampling for natural phrasing
        repetition_penalty=1.12,      # Prevents the model from repeating the same line over and over
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

prompt = "INT. A DIMLY LIT DINER - MIDNIGHT\n\n"

generated_script = generate_scene(prompt, max_len=250, temp=0.8)
print(generated_script)

--> Loading custom full-parameter model from /content/drive/MyDrive/Finetuning/output/final_screenplay_gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--> Model successfully loaded onto CUDA.

[PROMPT]: INT. A DIMLY LIT DINER - MIDNIGHT


INT. A DIMLY LIT DINER - MIDNIGHT

 
	As the diner is closed, a WINDOW comes to life in front of them...it's an old one-two punch from someplace else: The
	DAD'S CAR and THE DOOR TO THE BATHROOM we saw earlier! We see it on its own now as well; no doubt for two reasons : It has been the only place left open in this diner with a view out into town
	of every kind imaginable except death itself. Now all that remains are these three pictures taken in the diner by the SADIO - and this is where their bodyguards hang out until they are almost fully clothed but still have their arms outstretched. They get up and walk across traffic towards ...the bathroom door which leads inside here just as Mr Hirsch and Mr Grosse enter at the back entrance .
	There aren't any girls there yet either, so when somebody opens another door after them, they don’t even look at each other, then go inside again.
	And this time th

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

MODEL_PATH = "/content/drive/MyDrive/Finetuning/output/final_screenplay_gpt2"

print("--> Reloading local weights for Hub deployment...")
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)

HF_REPO_NAME = "raghavnimbalkar/gpt2-screenplay-generator"

print(f"--> Pushing weights to Hugging Face Hub: {HF_REPO_NAME}...")
model.push_to_hub(HF_REPO_NAME)
tokenizer.push_to_hub(HF_REPO_NAME)

print("\n--> Success! Your model is now publicly (or privately) hosted on the Hugging Face Hub.")
print(f"View it here: https://huggingface.co/{HF_REPO_NAME}")

--> Reloading local weights for Hub deployment...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

--> Pushing weights to Hugging Face Hub: raghavnimbalkar/gpt2-screenplay-generator...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zsfqea6/model.safetensors:   1%|          | 3.85MB /  498MB            

README.md: 0.00B [00:00, ?B/s]


--> Success! Your model is now publicly (or privately) hosted on the Hugging Face Hub.
View it here: https://huggingface.co/raghavnimbalkar/gpt2-screenplay-generator


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

MODEL_PATH = "/content/drive/MyDrive/Finetuning/output/final_screenplay_gpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT2LMHeadModel.from_pretrained(MODEL_PATH).to(device)
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)

def test_generation(prompt_title, prompt_text, max_tokens=450, temperature=0.82):
    print(f"\n=============================================")
    print(f"TEST GENRE: {prompt_title}")
    print(f"=============================================")
    print(f"[PROMPT]: {prompt_text.strip()}\n")

    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=max_tokens,       # Pushing close to our 512 training limit for longer output
        temperature=temperature,     # 0.82 balances creative vocabulary and screenplay formatting safety
        top_k=40,                    # Filters out low-probability nonsense words
        top_p=0.95,                  # Nucleus sampling for natural dialogue pacing
        repetition_penalty=1.15,     # Aggressively stops the model from looping names or indent actions
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_scene = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(generated_scene)
    print("\n" + "-"*45)

# --- PROMPT TEST SUITE ---

sci_fi_prompt = """
EXT. DESERT WASTELAND - NIGHT

The wind howls across empty sand dunes. A lone figure, MAX (30s), stumbles forward holding a glowing neon tracking device.

The device suddenly BEEPS violently.
"""

drama_prompt = """
INT. POLICE INTERROGATION ROOM - DAY

A single hanging light bulb sways. DETECTIVE MILLER sits across from Sarah.

MILLER
(leaning forward)
You expect me to believe you didn't see who took the money?
"""

indie_prompt = """
EXT. CITY ROOFTOP - DAWN

The sky is a pale purple. JAKE sits on the edge of the roof, dangling his feet over the massive drop.
"""

test_generation("SCI-FI SUSPENSE", sci_fi_prompt)
test_generation("TENSE DRAMA INTERROGATION", drama_prompt, temperature=0.75) # Lower temp for sharper dialogue
test_generation("INDIE COMING-OF-AGE", indie_prompt)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


TEST GENRE: SCI-FI SUSPENSE
[PROMPT]: EXT. DESERT WASTELAND - NIGHT

The wind howls across empty sand dunes. A lone figure, MAX (30s), stumbles forward holding a glowing neon tracking device. 

The device suddenly BEEPS violently.


EXT. DESERT WASTELAND - NIGHT

The wind howls across empty sand dunes. A lone figure, MAX (30s), stumbles forward holding a glowing neon tracking device. 

The device suddenly BEEPS violently.
BEN JORDAN's eyes are filled with fear and confusion as he runs out of space to find himself in the depths ahead...

	MAX'S POV: THE SURFACE OF LAUNCHER TWO ... He sees his way through dense forested grassland -- still frozen between trees; no sign that anything is up there yet! The ship moves along its path silently toward us from behind where we were standing before Max finally appears at an old tunnel entrance. His shadow falls on top him revealing two enormous figures which float above towering trees like giant balloons with one side attached below the other. Sud

In [ ]:
!pip freeze > /content/drive/MyDrive/Finetuning/output/cuda_requirements.txt
print("--> Package ecosystem locked and saved to Google Drive.")

--> Package ecosystem locked and saved to Google Drive.
